# Tiff Writer Test

Test the `TiffWriter` backend with a Micro-Manager demo acquisition.
All raw images, segmentation masks, and tracked labels are stored in a
single TIFF store.

## 1. Connect to microscope

In [1]:
from faro.microscope.demo import MMDemo

mic = MMDemo(micromanager_path="C:\\Program Files\\Micro-Manager-2.0")

In [2]:
import faro.core.utils as utils

utils.print_configs(mic.mmc)

# Check image dimensions from the demo camera
mic.mmc.snapImage()
test_img = mic.mmc.getImage()
print(f"Camera: {test_img.shape[1]}x{test_img.shape[0]}, dtype={test_img.dtype}")

Config Groups
├── Camera
│   ├── HighRes
│   ├── LowRes
│   └── MedRes
├── Channel
│   ├── Cy5
│   ├── DAPI
│   ├── FITC
│   └── Rhodamine
├── Channel-Multiband
│   ├── Cy5
│   ├── DAPI
│   ├── FITC
│   └── Rhodamine
├── LightPath
│   ├── Camera-left
│   ├── Camera-right
│   └── Eyepiece
├── Objective
│   ├── 10X
│   ├── 20X
│   └── 40X
└── System
    └── Startup

Camera: 512x512, dtype=uint16


## 2. Set up the pipeline

In [3]:
import os
import tempfile

from faro.core.data_structures import SegmentationMethod
from faro.segmentation.base import OtsuSegmentator
from faro.tracking.trackpy import TrackerTrackpy
from faro.feature_extraction.simple import SimpleFE
from faro.core.pipeline import ImageProcessingPipeline

path = os.path.join(tempfile.gettempdir(), "rtm-ome-zarr-test")

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=OtsuSegmentator(),
        use_channel=0,
        save_tracked=True,
    )
]

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=SimpleFE("labels"),
    tracker=TrackerTrackpy(),
)

Directory C:\Users\Alex\AppData\Local\Temp\rtm-ome-zarr-test\tracks already exists


## 3. Create the OmeZarrWriter

In [4]:
from faro.core.writers import TiffWriter

img_h, img_w = test_img.shape

writer = TiffWriter(storage_path=path)

## 4. Define experiment

In [5]:
from faro.core.data_structures import RTMSequence

seq = RTMSequence(
    time_plan={"interval": 1.0, "loops": 10},
    stage_positions=[
        {"x": 0.0, "y": 0.0, "z": 0.0},
    ],
    channels=[
        {"config": "DAPI", "exposure": 50},
        {"config": "FITC", "exposure": 100},
    ],
)

events = list(seq)
print(f"{len(events)} events")

10 events


## 5. Run experiment with OME-Zarr writer

In [6]:
from faro.core.controller import Controller

ctrl = Controller(mic, pipeline, writer=writer)
ctrl.run_experiment(events)
ctrl.finish_experiment()

print("Experiment complete.")

[04/01/26 12:48:39] INFO     MDA Started: GeneratorMDASequence()                                     ]8;id=809786;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=570283;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#378\378]8;;\

                    INFO     index={'t': 0, 'p': 0, 'c': 0} channel=Channel(config='DAPI')           ]8;id=444791;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=997601;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#337\337]8;;\
                             exposure=50.0 min_start_time=0.0 x_pos=0.0 y_pos=0.0 z_pos=0.0                        
                             metadata={'fov': 0, 'timestep': 0, 'fname': '000_00000', 'time': 0,                   
                             'stim': False, 'channels': ['DAPI', 'FITC'], 'img_type':                              
                             <ImgType.IMG_RAW: 1>}                                                                 

                    INFO     index={'t': 0, 'p': 0, 'c': 1} channel=Channel(config='FITC')           ]8;id=681959;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=575149;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#337\337]8;;\
                             exposure=100.0 min_start_time=0.0 z_pos=0.0 metadata={'fov': 0,                       
                             'timestep': 0, 'fname': '000_00000', 'time': 0, 'stim': False,                        
                             'channels': ['DAPI', 'FITC'], 'img_type': <ImgType.IMG_RAW: 1>}                       

[04/01/26 12:48:40] INFO     index={'t': 1, 'p': 0, 'c': 0} channel=Channel(config='DAPI')           ]8;id=122352;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=427006;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#337\337]8;;\
                             exposure=50.0 min_start_time=1.0 x_pos=0.0 y_pos=0.0 z_pos=0.0                        
                             metadata={'fov': 0, 'timestep': 1, 'fname': '000_00001', 'time': 1.0,                 
                             'stim': False, 'channels': ['DAPI', 'FITC'], 'img_type':                              
                             <ImgType.IMG_RAW: 1>}                                                                 

                    INFO     index={'t': 1, 'p': 0, 'c': 1} channel=Channel(config='FITC')           ]8;id=12943;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=156631;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#337\337]8;;\
                             exposure=100.0 min_start_time=1.0 z_pos=0.0 metadata={'fov': 0,                       
                             'timestep': 1, 'fname': '000_00001', 'time': 1.0, 'stim': False,                      
                             'channels': ['DAPI', 'FITC'], 'img_type': <ImgType.IMG_RAW: 1>}                       

...

                    INFO     index={'t': 8, 'p': 0, 'c': 1} channel=Channel(config='FITC')           ]8;id=168114;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=64073;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#337\337]8;;\
                             exposure=100.0 min_start_time=8.0 z_pos=0.0 metadata={'fov': 0,                       
                             'timestep': 8, 'fname': '000_00008', 'time': 8.0, 'stim': False,                      
                             'channels': ['DAPI', 'FITC'], 'img_type': <ImgType.IMG_RAW: 1>}                       

[04/01/26 12:48:48] INFO     index={'t': 9, 'p': 0, 'c': 0} channel=Channel(config='DAPI')           ]8;id=531232;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=270000;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#337\337]8;;\
                             exposure=50.0 min_start_time=9.0 x_pos=0.0 y_pos=0.0 z_pos=0.0                        
                             metadata={'fov': 0, 'timestep': 9, 'fname': '000_00009', 'time': 9.0,                 
                             'stim': False, 'channels': ['DAPI', 'FITC'], 'img_type':                              
                             <ImgType.IMG_RAW: 1>}                                                                 

                    INFO     index={'t': 9, 'p': 0, 'c': 1} channel=Channel(config='FITC')           ]8;id=906189;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=611317;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#337\337]8;;\
                             exposure=100.0 min_start_time=9.0 z_pos=0.0 metadata={'fov': 0,                       
                             'timestep': 9, 'fname': '000_00009', 'time': 9.0, 'stim': False,                      
                             'channels': ['DAPI', 'FITC'], 'img_type': <ImgType.IMG_RAW: 1>}                       

                    INFO     MDA Finished: GeneratorMDASequence()                                    ]8;id=378565;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=966271;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#465\465]8;;\

Experiment complete.
